# Heartsteps Matrix Completion using Nearest Neighbor Strategies

In [4]:
import numpy as np
import pandas as pd

In [5]:
# read in Heartsteps dataset
df_hs = pd.read_csv("data_use.csv")
df_hs

,user.index,date,time,5min_datetime,steps,rolling_sum,rt,sugg.decision.5min,send.sedentary,availability
0,1,7/22/2015,9:05:00,NaN,0,0,1,NaN,0,1
1,1,7/22/2015,9:10:00,NaN,0,0,1,NaN,0,1
2,1,7/22/2015,9:15:00,NaN,0,0,1,NaN,0,1
3,1,7/22/2015,9:20:00,NaN,0,0,1,NaN,0,1
4,1,7/22/2015,9:25:00,NaN,0,0,1,NaN,0,1
...,...,...,...,...,...,...,...,...,...,...
228235,37,1/26/2016,20:40:00,NaN,0,0,1,NaN,0,1
228236,37,1/26/2016,20:45:00,NaN,0,0,1,NaN,0,1
228237,37,1/26/2016,20:50:00,NaN,0,0,1,NaN,0,1
228238,37,1/26/2016,20:55:00,NaN,0,0,1,NaN,0,1


## Define `transform_dnn`

In [67]:
def transform_dnn(df, day_partitions = 4, day_measure = 144, units = 37):
    """
    Transfom df into shape suitable for DNN use case.

    Parameters
    ----------
    df : pandas DataFrame
         a matrix of size Z x 10 (with a set of properties to be defined)
         ...
    day_partitions : int
        The number of partitions to split an individual day into. The 
        default value is 4. Maximally, the value of day_partitions is 
        day_measure.
    day_measure : int
        The number of measurements per "day" in df. The default is 144,
        which aligns with the measurements per day in the HeartSteps dataset.
    units : int
        The number of units in df. Default is 37 (in line with the HeartSteps dataset)        
    Returns
    ----------
    tuple (M, A)
    where M is a units x (day_partitions * D) x (day_measure / day_partitions) np array 
    that contains the outcome vectors of interest and A is a np array shape units x (day_partitions * D)
    that indicates in which unit/time combination a treatment was observed.
    D is the maximum length of days observed among all unit trials.
    """
    # first, find the maximum trial length in days
    # units are 1-indexed
    max_days = 0
    num_days = []
    for i in range(1, units + 1):
        len_i = len(df[df['user.index'] == i]) // day_measure
        num_days.append(len_i)
        if len_i > max_days:
            max_days = len_i
    # init M and A
    M = np.zeros((units, day_partitions * max_days, day_measure // day_partitions))
    A = np.zeros((units, day_partitions * max_days))
    
    for i in range(units):
        # note: i is used to index into M and A.
        # unit_ind is used to retrieve unit specific values from df
        unit_ind = i + 1
        
        # slice by unit
        df_unit = df.loc[df["user.index"] == unit_ind]
        
        # pad first to max trial length to maintain partition size across units
        trial_steps = df_unit["steps"].to_numpy()
        trial_trt = df_unit["treatment"].to_numpy()
        if num_days[i] < max_days:
            # pad with 0s, step values are masked by trt
            pad = np.zeros((max_days - num_days[i]) * day_measure)
            trial_steps = np.append(trial_steps, pad)
            pad_trt = np.zeros((max_days - num_days[i]) * day_measure)
            trial_trt = np.append(trial_trt, pad_trt)
            
        # first split into days, then into partitions
        lst_days = np.split(trial_steps, max_days)
        parts = np.array([np.split(x, day_partitions) for x in lst_days])
         
        # Shape of parts is max_days x day_partitions x (day_measure // day_partitions)
        # Flatten along d dimension to achieve (day_partitions * max_days) x (day_measure // day_partitions) shape
        unit_parts = parts.reshape((max_days * day_partitions, day_measure // day_partitions))
        
        # Repeat for treatment col. 
        lst_days_trt = np.split(trial_trt, max_days)
        parts_trt = np.array([np.split(x, day_partitions) for x in lst_days_trt])
        unit_trt = parts_trt.reshape((max_days * day_partitions, day_measure // day_partitions))
        
        # if any of the trt values within the partition is 1, set the value in the final array to 1
        unit_trt_agg = unit_trt.any(axis = 1).astype(int)
        
        M[i] = unit_parts
        A[i] = unit_trt_agg
        A = A.astype('int32')
    return M, A
    

Question: could padding 0s to max trial length cause problems? i.e. people who have longer trials might be artificially more similar to each other compared to participants with shorter trial dates who now have no activity after the end of their trial date. 

## Testing `transform_dnn` with simulated symmetrical data

We create a small test dataset as a proof of concept of the above data transformation. This data set will have 3 units over a time span of 4 days, with 12 measurements a day. We partition each day into 3 segments (4 measurements per partition). The treatment is randomly applied across each day and time span. This is a simpler example than the actual use case, as the number of days within the trial is consistent for all users. See the section below for a more complex example.

In [69]:
# create test set

# user inds
u1 = np.repeat(1, 12 * 4)
u2 = np.repeat(2, 12 * 4)
u3 = np.repeat(3, 12 * 4)
user_index = np.concatenate((u1, u2, u3))

# step measurements (for simplicity, between 1 and 10)
rng = np.random.default_rng()
u1_steps = rng.integers(low=1, high=10, size=12 * 4)
u2_steps = rng.integers(low=1, high=10, size=12 * 4)
u3_steps = rng.integers(low=0, high=10, size=12 * 4)
steps = np.concatenate((u1_steps, u2_steps, u3_steps))
 
# treatment measurements
# in practice, it seems much more likely to see no treatment rather than a treatment
u1_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
u2_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
u3_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
trt = np.concatenate((u1_trt, u2_trt, u3_trt))

# create final dataframe
final_np = np.stack((user_index, steps, trt), axis = 1)
df_symtest = pd.DataFrame(final_np, columns = ["user.index", "steps", "treatment"])
df_symtest

,user.index,steps,treatment
0,1,8,0
1,1,7,0
2,1,6,0
3,1,3,1
4,1,5,0
...,...,...,...
139,3,8,1
140,3,3,0
141,3,6,0
142,3,2,1


Now that we have created our test set, we can use `transform_dnn` to demonstrate the desired transformation properties.

In [70]:
M, A = transform_dnn(df_symtest, day_partitions = 3, day_measure = 12, units = 3)

In [71]:
M

array([[[8., 7., 6., 3.],
        [5., 5., 5., 1.],
        [7., 9., 1., 2.],
        [9., 3., 1., 8.],
        [5., 3., 6., 1.],
        [9., 4., 8., 5.],
        [4., 6., 4., 9.],
        [4., 8., 6., 5.],
        [1., 8., 5., 3.],
        [1., 5., 4., 4.],
        [6., 3., 6., 9.],
        [8., 3., 7., 7.]],

       [[5., 6., 3., 8.],
        [8., 8., 6., 4.],
        [4., 2., 7., 9.],
        [3., 7., 3., 5.],
        [9., 9., 9., 3.],
        [4., 7., 3., 6.],
        [3., 3., 1., 1.],
        [4., 5., 6., 9.],
        [8., 1., 1., 4.],
        [6., 3., 2., 1.],
        [2., 4., 8., 1.],
        [2., 5., 6., 2.]],

       [[0., 7., 5., 9.],
        [2., 5., 9., 1.],
        [4., 7., 3., 5.],
        [9., 3., 7., 2.],
        [8., 1., 0., 6.],
        [7., 9., 5., 4.],
        [0., 7., 2., 7.],
        [2., 4., 0., 9.],
        [5., 5., 1., 8.],
        [6., 7., 3., 6.],
        [2., 3., 6., 8.],
        [3., 6., 2., 8.]]])

In [72]:
A

array([[1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 0, 1],
       [1, 1, 0, 0, 0, 1, 0, 1, 0, 1, 1, 1],
       [0, 0, 0, 0, 0, 1, 1, 0, 1, 1, 1, 1]])

## Testing `transform_dnn` on simulated asymmetrical data

Next, we test the function on a dataset with asymmetrical trial periods. As before, we will have 3 units, but this time, one of the units has a longer 6 day trial period. The remaining units have a 4 day trial period. There are still 12 measurements a day, and each day has 3 segments.

In [73]:
# create test set
# user inds, u1 has 6 day trial period
u1 = np.repeat(1, 12 * 6)
u2 = np.repeat(2, 12 * 4)
u3 = np.repeat(3, 12 * 4)
user_index = np.concatenate((u1, u2, u3))

# step measurements (for simplicity, between 1 and 10)
rng = np.random.default_rng()
u1_steps = rng.integers(low=1, high=10, size=12 * 6)
u2_steps = rng.integers(low=1, high=10, size=12 * 4)
u3_steps = rng.integers(low=0, high=10, size=12 * 4)
steps = np.concatenate((u1_steps, u2_steps, u3_steps))
 
# treatment measurements
# in practice, it seems much more likely to see no treatment rather than a treatment
u1_trt = np.random.choice([0, 1], 12 * 6, p=[0.8, 0.2])
u2_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
u3_trt = np.random.choice([0, 1], 12 * 4, p=[0.8, 0.2])
trt = np.concatenate((u1_trt, u2_trt, u3_trt))

# create final dataframe
final_np = np.stack((user_index, steps, trt), axis = 1)
df_asymtest = pd.DataFrame(final_np, columns = ["user.index", "steps", "treatment"])
df_asymtest

,user.index,steps,treatment
0,1,6,0
1,1,1,0
2,1,7,0
3,1,6,1
4,1,9,1
...,...,...,...
163,3,2,0
164,3,2,1
165,3,9,0
166,3,3,0


In [74]:
M_a, A_a = transform_dnn(df_asymtest, day_partitions = 3, day_measure = 12, units = 3)

In [75]:
M_a

array([[[6., 1., 7., 6.],
        [9., 7., 6., 4.],
        [8., 6., 8., 5.],
        [3., 9., 8., 5.],
        [2., 3., 8., 8.],
        [8., 3., 2., 1.],
        [4., 1., 3., 3.],
        [8., 4., 5., 8.],
        [6., 5., 9., 6.],
        [6., 2., 2., 8.],
        [4., 8., 9., 9.],
        [9., 4., 7., 4.],
        [2., 4., 9., 3.],
        [5., 7., 9., 7.],
        [3., 2., 7., 3.],
        [1., 6., 8., 1.],
        [4., 2., 9., 1.],
        [6., 5., 1., 2.]],

       [[4., 2., 6., 2.],
        [8., 3., 1., 7.],
        [8., 9., 9., 5.],
        [2., 4., 2., 1.],
        [7., 1., 3., 8.],
        [4., 2., 5., 2.],
        [7., 4., 7., 5.],
        [1., 3., 9., 4.],
        [1., 8., 1., 5.],
        [8., 4., 8., 8.],
        [9., 2., 1., 6.],
        [3., 5., 2., 2.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.],
        [0., 0., 0., 0.]],

       [[5., 3., 4., 2.],
        [2., 2., 4., 1.],
        

In [76]:
A_a

array([[1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1],
       [1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0],
       [0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0]])

## Test `transform_dnn` on HeartSteps dataset

Finally, we test `transform_dnn` on its primary use case, the HeartSteps dataset. We use `send.sedentary` as the "treatment" indicator.

In [77]:
df_hs["treatment"] = df_hs["send.sedentary"]

We can see that there are 37 units. Time statistics run from 9:05 to 21:00 at 5 minute intervals, yielding 12 observations per hour for 12 hours. This means that there are 144 measurements per day. We will demonstrate by partitioning the data into 4 sections per day (i.e. each section has 36 measurements). 

In [81]:
# note/question: it seems that 4 measurements may be too granular to find enough neighbors
M_hs, A_hs = transform_dnn(df_hs, day_partitions = 4, day_measure = 144, units = 37)

In [82]:
with np.printoptions(threshold=np.inf):
    print(M_hs[0, :, :])

[[  0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.   0.
    0.   0.   0.   0.   0.   0.   0.   0.  67. 115.  60.   0.  61.   0.
  169.   0.   0. 524. 324.   0.  26.   0.]
 [  0.   0.   0.   0.   0.   0.   0.   0. 212. 396. 457. 246.   0. 102.
    0.   0.   0.   0.   0.  38.   0.   0.  64. 102.   0. 123.  74. 221.
    0.   0. 260. 154.   0.   0.   0.   0.]
 [  0.   0.   0.   0.   0.   0.   0.  62.   0.  60. 170.  28. 326.  25.
    0.   0.   0.   0.   0.  92.  21.  57.   0.  25. 135.  22.   0.   0.
   72.  32.  70.  70. 108.  68.  25.   0.]
 [ 83.  31.  18.   0.   0.   0.  93.  33.  56. 135. 365. 580. 584. 584.
  576. 558. 523.  88. 229.   0.   0.   0.   0. 140.   0.   0.   0.  92.
   27. 227.   0.   0.   0. 156. 122.  89.]
 [590. 587. 279.   4. 113. 131.   0.   0.   0.   0.   0.   0. 136. 101.
   12.  89. 129.  60. 148.  81.  65.   0.  44.   0.  88. 196.  50.  54.
   89. 210.  52. 194. 143. 121. 294. 126.]
 [206. 147. 277. 350. 119. 147. 157.   0. 183. 349. 204.   0. 173

In [83]:
A_hs

array([[0, 0, 1, ..., 0, 0, 0],
       [0, 0, 1, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]])

## Li 2019 Nearest Neighbors Implementation

We implement the user-user nearest neighbors algorithm as defined in Nearest Neighbors for Matrix Estimation Interpreted as Blind Regression for Latent Variable Model (Li et al. 2019). More specifically, we construct Algorithm 2, which can be found in Section III [here](https://arxiv.org/abs/1705.04867).

First, to align the NN implementation with the data format considered above, we define a function to collapse the partitioned measurements dimension (3rd dim) to its mean.

In [91]:
def mean_collapse(A, d = 2):
    """
    Returns A (3d array) with dth dimension collapsed as mean
    """
    return np.mean(A, axis = d)

Next, we construct the necessary functions as defined in Li 2019.

In [90]:
def dissim_row(O, M, u, v):
    """
    Returns the row dissimilarity between rows u and v
    in O
    
    Parameters
    ----------
    O : a matrix of size a x b
    M : a masking matrix of size a x b that
        dictates which entries of O are known 
    u : the index of a row in O
    v : the index of a row in O

    Returns
    ----------
    d_row : the row dissimilarity between u and v (float)
    """
    obv_col_mask = M[u] * M[v]
    overlap = np.sum(obv_col_mask)
    if not overlap:
        return float('inf')    
    sd = (O[u] - O[v])**2 * obv_col_mask 
    msd = (1 / overlap) * np.sum(sd)
    return msd

def nearest_neighbors(O_1, u, M, eta):
    """
    Returns the B neighborbood around u using rows in O_1 within
    radius eta. 
    
    Parameters
    ----------
    O_1 : a matrix of size c x d
    M : a masking matrix of size c x d that
        dictates which entries of O_1 are known 
    eta : radius of B neighborhood
    
    Returns
    ----------
    B : neighborhood (indicies of most similar rows)
        with radius eta
    """
    dissims = np.array([dissim_row(O_1, M, u, v) for v in range(len(O_1))])
    # remove u from neighbors
    #dissims = np.delete(dissims, u, 1)
    B = np.nonzero(dissims <= eta)
    return B
    
    
    
def estimate_li_nn(Z, M, eta):
    """
    Returns A_hat, the estimate of partially unobvserved
    matrix A using Nearest Neighbors imputation
    
    Parameters
    ----------
    Z : an array of size m x n x q
    M : a masking matrix of size m x n that
        dictates which entries of Z are known 
    eta : radius of B neighborhood
    
    Returns
    ----------
    A_hat : the estimate of partially unobvserved matrix A
    """
    Z = mean_collapse(Z, d = 2)
    m, n = Z.shape
    A_hat = np.zeros((m, n))
    for u in range(m):
        for v in range(n):
            O_1  = np.delete(Z, v, axis = 1)
            O_2 = Z[:, v]
            M_v = np.delete(M, v, axis = 1)
            u_neighbors = nearest_neighbors(O_1, u, M_v, eta)
            # only select obvserved neighbors
            mask = M[u_neighbors, v]
            pos_nn = Z[u_neighbors, v]
            masked_nn = pos_nn[mask == 1]
            if not len(masked_nn):
                A_hat[u, v] = 0
            else:
                A_hat[u, v] = np.mean(masked_nn)
    return A_hat

def estimate_li_nn_inds(Z, M, eta, v_inds):
    """
    Returns A_hat, the estimate of partially unobvserved
    matrix A using Nearest Neighbors imputation, with only indices specified
    by v_inds estimated
    
    Parameters
    ----------
    Z : an array of size m x n x q
    M : a masking matrix of size m x n that
        dictates which entries of Z are known 
    eta : radius of B neighborhood
    v_inds : indices to estimate, tuple of (x_coords, y_coords)
    
    Returns
    ----------
    A_hat : the estimate of values in partially unobserved matrix A specified by v_inds
    """
    row_coords, col_coords = v_inds
    Z = mean_collapse(Z, d = 2)
    m, n = Z.shape
    A_hat = np.zeros((m, n))
    for i in range(len(row_coords)):
        u = row_coords[i]
        j = col_coords[i]
        O_1  = np.delete(Z, j, axis = 1)
        O_2 = Z[:, j]
        M_j = np.delete(M, j, axis = 1)
        u_neighbors = nearest_neighbors(O_1, u, M_j, eta)
        mask = M[u_neighbors, j]
        pos_nn = Z[u_neighbors, j]
        masked_nn = pos_nn[mask == 1]
        if not len(masked_nn):
            A_hat[u, j] = 0
        else:
            A_hat[u, j] = np.mean(masked_nn)
    return A_hat
        
# using the indexing strategy i am currently using, A_hat and A are both 1 dimensional. This might change.
def mse(A_hat, A):
    """
    Returns the mean squared error of estimate A_hat from ground truth A
    """
    return np.mean((A_hat - A)**2)

### Checking Li 2019 implementation correctness with worked example

In order to look for bugs in our cross validation and Li 2019 implementations, we create a sample dataset with some nice properties and compute the similarities by hand. This doesn't guarantee correctness, but it will catch simple errors/mistakes in understanding.

To start, we create a simulated datasest with 6 users over a two day trial period. There are 6 measurements per day and each day is broken into 3 segments. 3 of the users (A, B, and C) are similar to each other (roughly more steps in the evening/last measurement periods in the day) and the other 3 users (D, E, F) are different from the first 3 and similar to each other (roughly more steps in the morning/early measurement periods.  

Let's suppose the matrix is fully observed. To simulate a validation set, we mask out one segement for each user - these are the values we will try to impute.

In [28]:
# user inds
uA = np.repeat(1, 6 * 2)
uB = np.repeat(2, 6 * 2)
uC = np.repeat(3, 6 * 2)

uD = np.repeat(4, 6 * 2)
uE = np.repeat(5, 6 * 2)
uF = np.repeat(6, 6 * 2)

user_index = np.concatenate((uA, uB, uC, uD, uE, uF))

# step measurements
steps_uA = [1, 1, 2, 3, 5, 9, 1, 1, 3, 4, 5, 10]
steps_uB = [0, 2, 2, 4, 5, 11, 0, 2, 3, 3, 4, 12]
steps_uC = [0, 0, 2, 5, 7, 10, 2, 2, 1, 5, 6, 9]

steps_uD = [10, 5, 4, 3, 1, 1, 9, 5, 3, 2, 1, 1]
steps_uE = [12, 4, 3, 3, 2, 0, 11, 5, 4, 2, 2, 0]
steps_uF = [9, 6, 5, 1, 2, 2, 10, 7, 5, 2, 0, 0]
steps = np.concatenate((steps_uA, steps_uB, steps_uC, steps_uD, steps_uE, steps_uF))

# treatment indicators
trt_uA = [1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0]
trt_uB = [0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0]
trt_uC = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]

trt_uD = [1, 0, 1, 0, 0, 0, 1, 1, 0, 1, 1, 0]
trt_uE = [0, 0, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0]
trt_uF = [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0]

trt = np.concatenate((trt_uA, trt_uB, trt_uC, trt_uD, trt_uE, trt_uF))
final_np = np.stack((user_index, steps, trt), axis = 1)
df_nn_test = pd.DataFrame(final_np, columns = ["user.index", "steps", "treatment"])
df_nn_test


,user.index,steps,treatment
0,1,1,1
1,1,1,0
2,1,2,1
3,1,3,0
4,1,5,0
...,...,...,...
67,6,7,1
68,6,5,1
69,6,2,1
70,6,0,0


In [51]:
data_full, mask = transform_dnn(df_nn_test, day_partitions = 3, day_measure = 6, units = 6)

In [52]:
data_full

array([[[ 1.,  1.],
        [ 2.,  3.],
        [ 5.,  9.],
        [ 1.,  1.],
        [ 3.,  4.],
        [ 5., 10.]],

       [[ 0.,  2.],
        [ 2.,  4.],
        [ 5., 11.],
        [ 0.,  2.],
        [ 3.,  3.],
        [ 4., 12.]],

       [[ 0.,  0.],
        [ 2.,  5.],
        [ 7., 10.],
        [ 2.,  2.],
        [ 1.,  5.],
        [ 6.,  9.]],

       [[10.,  5.],
        [ 4.,  3.],
        [ 1.,  1.],
        [ 9.,  5.],
        [ 3.,  2.],
        [ 1.,  1.]],

       [[12.,  4.],
        [ 3.,  3.],
        [ 2.,  0.],
        [11.,  5.],
        [ 4.,  2.],
        [ 2.,  0.]],

       [[ 9.,  6.],
        [ 5.,  1.],
        [ 2.,  2.],
        [10.,  7.],
        [ 5.,  2.],
        [ 0.,  0.]]])

In [53]:
mask

array([[1., 1., 0., 1., 1., 1.],
       [0., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 0.],
       [1., 1., 0., 1., 1., 1.],
       [0., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 0.]])

First, we take the mean of the partitioned dimension to achieve the 2D structure.

In [54]:
data = mean_collapse(data_full)
data

array([[1. , 2.5, 7. , 1. , 3.5, 7.5],
       [1. , 3. , 8. , 1. , 3. , 8. ],
       [0. , 3.5, 8.5, 2. , 3. , 7.5],
       [7.5, 3.5, 1. , 7. , 2.5, 1. ],
       [8. , 3. , 1. , 8. , 3. , 1. ],
       [7.5, 3. , 2. , 8.5, 3.5, 0. ]])

Next, let's calculate the dissimilarities between each row. This is the mean squared difference between overlapping observed values. 

Dissim A-B: $\frac{((3 - 2.5)^2 + (1 - 1)^2 + (3.5 - 3.0)^2 + (7.5 - 8)^2)}{4} = \frac{0.75}{4} = 0.1874$

Dissim A-C: $\frac{(1-0)^2 + (3.5 - 2.5)^2 + (1)^2 + (0.5)^2 + 0^2}{4} = \frac{3.25}{4} = 0.8125$

Dissim A-D: $\frac{6.5^2 + 1^2 + 6^2 + 1^2 + 6.5^2}{5} = \frac{122.5}{5} = 24.5$

Dissim A-E: $\frac{0.5^2 + 6^2 + 7^2 + 6.5^2}{4} = 31.875$

Dissim A-F: $\frac{6.5^ + 0.5^2 + 7.5^2 + 0}{4} = 24.6875$ 

Dissim B-C: $\frac{0.5^2 + 0.5^2 + 1^2 + 0^2}{4} = 0.375$

Dissim B-D: $\frac{0.5^2 + 6^2 + 0.5^2 + 7^2}{4} = 21.375$

Dissim B-E: $\frac{0^2 + 7^2 + 7^2 + 0^2 + 7^2}{5} = 29.4$

Dissim B-F: $\frac{0^2 + 6^2 + 7.5^2 + 0.5^2}{4} = 23.125$

Dissim C-D: $\frac{7.5^2 + 0^2 + 5^2 + 0.5^2}{4} = 20.375$

Dissim C-E: $\frac{0.5^2 + 7^2 + 7^2 + 0^2}{4} = 24.5625$

Dissim C-F: $\frac{7.5^2 + 0.5^2 + 6.5^2 + 6.5^2 + 0.5^2}{5} = 28.25$

Dissim D-E: $\frac{0.5^2 + 1^2 + 0.5^2 + 0^2}{4} = 0.375$

Dissim D-F: $\frac{0^2 + 0.5^2 + 0.5^2 + 1^2}{4} = 0.375$

Dissim E-F: $\frac{0^2 + 1^2 + 0.5^2 + 0.5^2}{4} = 0.375$

We can see that A-B-C and D-E-F are two distinct neighborhoods with $\eta = 1$. Given this, we impute the missing values by taking the average of the neighbors at the column where an observation is missing. 

In [55]:
inv_mask = ~(mask.astype(bool))
mask


array([[1., 1., 0., 1., 1., 1.],
       [0., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 0.],
       [1., 1., 0., 1., 1., 1.],
       [0., 1., 1., 1., 1., 1.],
       [1., 1., 1., 1., 1., 0.]])

In [56]:
data

array([[1. , 2.5, 7. , 1. , 3.5, 7.5],
       [1. , 3. , 8. , 1. , 3. , 8. ],
       [0. , 3.5, 8.5, 2. , 3. , 7.5],
       [7.5, 3.5, 1. , 7. , 2.5, 1. ],
       [8. , 3. , 1. , 8. , 3. , 1. ],
       [7.5, 3. , 2. , 8.5, 3.5, 0. ]])

In [57]:
data_imputed = data.copy()
data_imputed[0, 2] = 8.25
data_imputed[1, 0] = 0.5
data_imputed[2, 5] = 7.75
data_imputed[3, 2] = 1.5
data_imputed[4, 0] = 7.5
data_imputed[5, 5] = 1

In [44]:
# these are the imputed predictions
pred = data_imputed[inv_mask]
pred

array([8.25, 0.5 , 7.75, 1.5 , 7.5 , 1.  ])

In [84]:
# this is the ground truth
truth = data[inv_mask]
truth

array([7. , 1. , 7.5, 1. , 8. , 0. ])

In [85]:
# the mse is:
mse(pred, truth)

0.5625

When we compute the MSE of the prediction by hand following Li 2019's method, we get 0.5625. Now we compute it using our implementation.

In [86]:
inds = np.nonzero(inv_mask)
A_hat = estimate_li_nn_inds(data_full, mask, eta = 1, v_inds = inds)
pred = A_hat[inds]
mse(pred, truth)

0.5625

## Cross Validation for $\eta$

Next, we implement cross-validation for the $\eta$ radius parameter. Given a dataset, an error metric, and an estimation function, we evaluate a range of values of $\eta$. 

In [ ]:
# notes
    # we are only interested in the estimates that `estimate` makes for the
    # validation set in particular. For any validation fold, we must only consider observed values (need a ground truth)
    # We mask over these values to estimate, then extract the estimates using the original mask to compute the error.

In [89]:
def cross_validate_nn(Z, M, k, eta, estimate, error_metric):
    """
    Performs k-fold cross validation on using dataset `Z`, mask `M`, parameters `k` and `eta`,
    estimation function `estimate`, and error `error_metric`
    """
    # find observed indices and create splits off of those
    obvs_inds_x, obvs_inds_y = np.nonzero(M)
    
    # shuffle the x and y coordinates in the same way to maintain valid coords
    shuf_inds = np.arange(len(obvs_inds_x))
    np.random.shuffle(shuf_inds)
    shuf_obvs_inds = np.array((obvs_inds_x[shuf_inds], obvs_inds_y[shuf_inds]))
    # split coordinates into k-folds
    folds = np.array_split(shuf_obvs_inds, k, axis = 1)
    tot_error = 0
    for i in range(k):
        # indices of the observed outcomes in the validation set in this iteration
        folds_ind = tuple(folds[i])
        # ground truth
        val_data = mean_collapse(Z[folds_ind], d = 1)
        est_mask = M.copy()
        # hide observed values in the validation set
        est_mask[folds_ind] = 0
        A_hat = estimate(Z, est_mask, eta, folds_ind)
        # extract validation points
        est_val = A_hat[folds_ind]
        tot_error += error_metric(est_val, val_data)
    # return average error over k folds
    return tot_error / k
 

### Cross validation for Li (2019) NN

We use k-fold cross validation to tune hyperparameter $\eta$ for the Li (2019) NN model implemented above. We show results using k = 5 and k = # observed outcomes (LOOCV). 

In [14]:
# first, compute dissims from a subsection of the data to get a sense of the range of distances
# notation: should change later, but M_hs is Z and A_hs is M
from itertools import product
rnd_ind = np.random.choice(37, size = 10, replace = False)
all_combs = product(rnd_ind, rnd_ind)
M_hs_collapse = mean_collapse(M_hs)
for u, v in all_combs:
    print(dissim_row(M_hs_collapse, A_hs, u, v))

0.0
1663.7362075617282
9433.312114197533
10708.135159465019
3178.353780864197
3858.7409979423865
2767.0774911816584
3845.4449588477364
2566.7926587301586
909.1473765432097
1663.7362075617282
0.0
3244.6373456790116
9885.76822916667
2104.8924575617284
4078.115162037037
1253.3985339506173
6343.9231701940025
2608.8954046639233
101.36844135802468
9433.312114197533
3244.6373456790116
0.0
13604.630401234566
1551.4845679012344
437.0987654320986
10156.326003086417
368.42669753086415
4308.5841049382725
64.90432098765433
10708.135159465019
9885.76822916667
13604.630401234566
0.0
3497.620627572016
7706.397376543211
460.83024691358025
6202.21965020576
9439.857561728395
13153.720061728394
3178.353780864197
2104.8924575617284
1551.4845679012344
3497.620627572016
0.0
2734.4951774691353
910.4014274691356
8328.937885802468
4025.8326581790125
5520.264274691359
3858.7409979423865
4078.115162037037
437.0987654320986
7706.397376543211
2734.4951774691353
0.0
8106.286779835391
1154.5992283950616
905.491126543

In [92]:
# try using HeartSteps data with eta 10 to 500? are there other hyperparameter optimizations to use? 
# k = 5
eta_range = np.arange(10, 510, step = 10)
#eta_range = [350.]

min_error = float("inf")
best_eta = 0
for eta in eta_range:
    error = cross_validate_nn(M_hs, A_hs, 5, eta, estimate_li_nn_inds, mse)
    print("Error for eta " + str(eta) + ": " + str(error))
    if error < min_error:
        min_error = error
        best_eta = eta


Error for eta 10: 4470.526945546737
Error for eta 20: 4475.536943415636
Error for eta 30: 4476.936157196134
Error for eta 40: 4471.450330926661
Error for eta 50: 4523.235176899617
Error for eta 60: 4479.808514651309
Error for eta 70: 4441.046044082525
Error for eta 80: 4482.697518328679
Error for eta 90: 4671.896733367627
Error for eta 100: 4684.731098630706
Error for eta 110: 4621.9614494813095
Error for eta 120: 4632.585121579831
Error for eta 130: 4604.341967029198
Error for eta 140: 4545.1265629439795
Error for eta 150: 4591.403307962594
Error for eta 160: 4602.311572546172
Error for eta 170: 4556.604497847468
Error for eta 180: 4553.882384075544
Error for eta 190: 4626.332304131758
Error for eta 200: 4542.30245564827
Error for eta 210: 4595.234190163875
Error for eta 220: 4546.3780593309075
Error for eta 230: 4553.129749538506
Error for eta 240: 4571.724398543994
Error for eta 250: 4586.534017165392
Error for eta 260: 4387.331505732779
Error for eta 270: 4388.360103636281
Error fo

In [87]:
# k = # observed entries
k = len(np.nonzero(A_hs)[0])
min_error = float("inf")
best_eta = 0
for eta in eta_range:
    error = cross_validate_nn(M_hs, A_hs, 50, eta, estimate_li_nn_inds, mse)
    print("Error for eta " + str(eta) + ": " + str(error))
    if error < min_error:
        min_error = error
        best_eta = eta

<class 'float'>
[[[  0.   0.   0. ...   0.  26.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...  68.  25.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  42.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  24.  18.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 ...

 [[  0.   0.   0. ...   0.   0. 296.]
  [218.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  22.  66.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   

[[[  0.   0.   0. ...   0.  26.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...  68.  25.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  42.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  24.  18.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 ...

 [[  0.   0.   0. ...   0.   0. 296.]
  [218.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  22.  66.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.

[[[  0.   0.   0. ...   0.  26.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...  68.  25.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  42.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  24.  18.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 ...

 [[  0.   0.   0. ...   0.   0. 296.]
  [218.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  22.  66.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.

[[[  0.   0.   0. ...   0.  26.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...  68.  25.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  42.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  24.  18.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 ...

 [[  0.   0.   0. ...   0.   0. 296.]
  [218.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  22.  66.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.

[[[  0.   0.   0. ...   0.  26.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...  68.  25.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  42.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  24.  18.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 ...

 [[  0.   0.   0. ...   0.   0. 296.]
  [218.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  22.  66.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.

...

[[[  0.   0.   0. ...   0.  26.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...  68.  25.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  42.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  24.  18.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 ...

 [[  0.   0.   0. ...   0.   0. 296.]
  [218.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  22.  66.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.

[[[  0.   0.   0. ...   0.  26.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...  68.  25.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  42.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  24.  18.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 ...

 [[  0.   0.   0. ...   0.   0. 296.]
  [218.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  22.  66.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.

[[[  0.   0.   0. ...   0.  26.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...  68.  25.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  42.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  24.  18.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 ...

 [[  0.   0.   0. ...   0.   0. 296.]
  [218.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  22.  66.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.

[[[  0.   0.   0. ...   0.  26.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...  68.  25.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  42.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  24.  18.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 ...

 [[  0.   0.   0. ...   0.   0. 296.]
  [218.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  22.  66.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.

[[[  0.   0.   0. ...   0.  26.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...  68.  25.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  42.   0.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  24.  18.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 ...

 [[  0.   0.   0. ...   0.   0. 296.]
  [218.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.  22.  66.]
  ...
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.   0. ...   0.   0.   0.]]

 [[  0.   0.   0. ...   0.   0.   0.]
  [  0.   0.

In [27]:
best_eta

440